--- Day 9: Disk Fragmenter ---
Another push of the button leaves you in the familiar hallways of some friendly amphipods! Good thing you each somehow got your own personal mini submarine. The Historians jet away in search of the Chief, mostly by driving directly into walls.

While The Historians quickly figure out how to pilot these things, you notice an amphipod in the corner struggling with his computer. He's trying to make more contiguous free space by compacting all of the files, but his program isn't working; you offer to help.

He shows you the disk map (your puzzle input) he's already generated. For example:

2333133121414131402
The disk map uses a dense format to represent the layout of files and free space on the disk. The digits alternate between indicating the length of a file and the length of free space.

So, a disk map like 12345 would represent a one-block file, two blocks of free space, a three-block file, four blocks of free space, and then a five-block file. A disk map like 90909 would represent three nine-block files in a row (with no free space between them).

Each file on disk also has an ID number based on the order of the files as they appear before they are rearranged, starting with ID 0. So, the disk map 12345 has three files: a one-block file with ID 0, a three-block file with ID 1, and a five-block file with ID 2. Using one character for each block where digits are the file ID and . is free space, the disk map 12345 represents these individual blocks:

0..111....22222
The first example above, 2333133121414131402, represents these individual blocks:

00...111...2...333.44.5555.6666.777.888899
The amphipod would like to move file blocks one at a time from the end of the disk to the leftmost free space block (until there are no gaps remaining between file blocks). For the disk map 12345, the process looks like this:

0..111....22222
02.111....2222.
022111....222..
0221112...22...
02211122..2....
022111222......
The first example requires a few more steps:

00...111...2...333.44.5555.6666.777.888899
009..111...2...333.44.5555.6666.777.88889.
0099.111...2...333.44.5555.6666.777.8888..
00998111...2...333.44.5555.6666.777.888...
009981118..2...333.44.5555.6666.777.88....
0099811188.2...333.44.5555.6666.777.8.....
009981118882...333.44.5555.6666.777.......
0099811188827..333.44.5555.6666.77........
00998111888277.333.44.5555.6666.7.........
009981118882777333.44.5555.6666...........
009981118882777333644.5555.666............
00998111888277733364465555.66.............
0099811188827773336446555566..............
The final step of this file-compacting process is to update the filesystem checksum. To calculate the checksum, add up the result of multiplying each of these blocks' position with the file ID number it contains. The leftmost block is in position 0. If a block contains free space, skip it instead.

Continuing the first example, the first few blocks' position multiplied by its file ID number are 0 * 0 = 0, 1 * 0 = 0, 2 * 9 = 18, 3 * 9 = 27, 4 * 8 = 32, and so on. In this example, the checksum is the sum of these, 1928.

Compact the amphipod's hard drive using the process he requested. What is the resulting filesystem checksum? (Be careful copy/pasting the input for this puzzle; it is a single, very long line.)

Your puzzle answer was 6607511583593.

--- Part Two ---
Upon completion, two things immediately become clear. First, the disk definitely has a lot more contiguous free space, just like the amphipod hoped. Second, the computer is running much more slowly! Maybe introducing all of that file system fragmentation was a bad idea?

The eager amphipod already has a new plan: rather than move individual blocks, he'd like to try compacting the files on his disk by moving whole files instead.

This time, attempt to move whole files to the leftmost span of free space blocks that could fit the file. Attempt to move each file exactly once in order of decreasing file ID number starting with the file with the highest file ID number. If there is no span of free space to the left of a file that is large enough to fit the file, the file does not move.

The first example from above now proceeds differently:

00...111...2...333.44.5555.6666.777.888899
0099.111...2...333.44.5555.6666.777.8888..
0099.1117772...333.44.5555.6666.....8888..
0099.111777244.333....5555.6666.....8888..
00992111777.44.333....5555.6666.....8888..
The process of updating the filesystem checksum is the same; now, this example's checksum would be 2858.

Start over, now compacting the amphipod's hard drive using this new method instead. What is the resulting filesystem checksum?

Your puzzle answer was 6636608781232.

Both parts of this puzzle are complete! They provide two gold stars: **

In [41]:
def generate_memory_map(p_initial_string):
    memory_map = {}
    ind = 0
    file_id = 0
    position = 0
    for i in range(len(p_initial_string)):
        if i%2 == 0:
            file_size = int(p_initial_string[i])
            for j in range(file_size):
                memory_map[position] = file_id
                position += 1
            file_id += 1
        else:
            space_chunk_size = int(p_initial_string[i])
            for j in range(space_chunk_size):
                memory_map[position] = -1
                position += 1        

    return memory_map

In [46]:
def find_first_free_space(p_map):
    for i in range(len(p_map)):
        if p_map[i] == -1:
            return i 
    return -1

In [79]:
def compress(p_map):
    for i in range(len(p_map)-1, -1, -1):
        first_free_block = find_first_free_space(p_map)
        if first_free_block >= 0 and first_free_block < i and p_map[i] >= 0:
            p_map[first_free_block] = p_map[i]
            p_map[i] = -1
        print('{}, '.format(i), end = '')

In [80]:
def calculate_checksum(p_map):
    s = 0
    for (i, j) in p_map.items():
        if j >= 0:
            s += i*j
    return s

In [ ]:
from copy import deepcopy

initial_string = ''

with open('input.txt', 'r') as file:
    initial_string = next(file).strip()

initial_memory_map = generate_memory_map(initial_string)

compressed_memory_map = deepcopy(initial_memory_map)

compress(compressed_memory_map)

checksum = calculate_checksum(compressed_memory_map)

print(checksum)

6607511583593


Part 1 answer: 6607511583593

In [138]:
def find_file_length(p_map, i):
    file_id = p_map[i]
    v_lenght = 0
    j = i
    while j >= 0 and p_map[j] == file_id:
        v_lenght += 1
        j -= 1
    return v_lenght

In [139]:
def find_first_free_space(p_map, p_file_length):
    i = 0
    while i < len(p_map):
        free_block = False
        block_len = 0    
        #print('Calculating position {}'.format(i))
        while i + block_len < len(p_map) and block_len < p_file_length and p_map[i + block_len] == -1:
            block_len += 1
        #print('block_len = {}'.format(block_len))
        if block_len > 0 and block_len >= p_file_length :
            return i
        else:
            i += 1

    return -1

In [140]:
def move_file(p_map, p_old_position, p_file_lenght, p_new_position):
    for i in range(p_file_lenght):
        p_map[p_new_position + i] = p_map[p_old_position + i]
    for i in range(p_file_lenght):
        p_map[p_old_position + i] = -1        

In [151]:
def print_map(p_map):
    print(''.join([str(x) if x != -1 else '.' for x in p_map.values() ]))

In [161]:
def compress_whole_files(p_map):
    i = len(p_map) - 1
    while i >= 0:
        if p_map[i] > -1:
            file_id = p_map[i]
            file_length = find_file_length(p_map, i)
            new_position = find_first_free_space(p_map, file_length)
            if new_position >= 0 and new_position < i:
                move_file(p_map, i - file_length + 1, file_length, new_position)
            i = i - file_length
        else:
            i -= 1
    print('Calculating block {}'.format(i), end = ', ')

In [163]:
from copy import deepcopy

initial_string = ''

with open('input.txt', 'r') as file:
    initial_string = next(file).strip()

initial_memory_map = generate_memory_map(initial_string)

compressed_memory_map = deepcopy(initial_memory_map)

#print('{}: '.format(len(compressed_memory_map)), end = '') 
#print_map(compressed_memory_map)

compress_whole_files(compressed_memory_map)

checksum = calculate_checksum(compressed_memory_map)

#vprint_map(compressed_memory_map)

print(checksum)

95450: 00...1111..222.....33333444.......555..66666.7777.888.9999999....101010101010...111111.......1212121212121212.131313131313131313........141414141414141414...15151515151515.......161616161616161616.171717...1818181818...19.......2020....2121....22232323.242424..25........2626......2727..28282828282828...29292929292929.........3030303030.3131.....323232323232323232........3333333333...34343434...35353535353535.........3636363636..37373737373737.........3838383838383838.393939393939393939....404040404040404040....41414141414141.......4242424242.43434343434343........4444444444....45.......4646464646464646........4747474747474747...484848.....494949494950505050........5151515151....5252525252.......535353........5454.....5555555555555555...56565656.5757...5858585858....59595959595959596060.........6161616161616161......6262......636363636363636363.........646464646464646464......6565656565........66666666666666..6767.686868.........69696969707070.71.72727272..73737373737373....74747

Part 2 result: 6636608781232